# Capa Bronze - CSV a Delta
Lee los CSV descomprimidos en el Volume y crea:
- `viajes_stm` (un lote por mes, se puede re-ejecutar: saltea los meses ya cargados)
- `horarios_por_parada`
- `horarios_puntos_control`

Todas las columnas se guardan como texto (string), los tipos se convierten despues.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema_bronze", "movilidad_bronze")
dbutils.widgets.text("schema_raw", "movilidad")
dbutils.widgets.text("volume", "raw")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema_bronze")
RAW_SCHEMA = dbutils.widgets.get("schema_raw")
VOLUME = dbutils.widgets.get("volume")

# Los archivos se extaren del Volume.
BASE = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{VOLUME}"

#crea el schema si  ya no  existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print("Archivos de origen:", BASE)
print("Tablas de destino:", f"{CATALOG}.{SCHEMA}")

In [ ]:
import os, re, csv, glob, unicodedata
from pyspark.sql import functions as F
from datetime import datetime

def limpiar_nombre(c):
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    c = re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower()
    return c or "col"


def detectar_formato(path):
    """Devuelve (separador, encoding) mirando el comienzo del archivo."""
    with open(path, "rb") as f:
        crudo = f.read(65536)
    try:
        texto = crudo.decode("utf-8")
        enc = "UTF-8"
    except UnicodeDecodeError:
        texto = crudo.decode("latin-1")
        enc = "ISO-8859-1"
    muestra = "\n".join(texto.splitlines()[:20])
    try:
        sep = csv.Sniffer().sniff(muestra, delimiters=",;|\t").delimiter
    except csv.Error as e:
        raise ValueError(
        f"No se pudo determinar el separador de {path}"
        ) from e
    return sep, enc


def leer_csv(archivos):
    if not archivos:
        raise ValueError("No se encontraron archivos CSV.")

    formatos = []
    encabezados = []

    for path in archivos:
        sep, enc = detectar_formato(path)
        formatos.append((sep, enc))

        with open(path, "r", encoding=enc, newline="") as f:
            encabezado = next(csv.reader(f, delimiter=sep), None)

        if not encabezado:
            raise ValueError(f"Archivo sin encabezado: {path}")

        encabezados.append(encabezado)

    if len(set(formatos)) != 1:
        raise ValueError(
            "Los CSV del lote tienen separadores o codificaciones diferentes."
        )

    if any(h != encabezados[0] for h in encabezados[1:]):
        raise ValueError(
            "Los CSV del lote tienen encabezados diferentes."
        )

    sep, enc = formatos[0]

    df = (
        spark.read
        .option("header", True)
        .option("sep", sep)
        .option("encoding", enc)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .csv(archivos)
    )

    nombres = [limpiar_nombre(c) for c in df.columns]

    if len(nombres) != len(set(nombres)):
        raise ValueError(
            "La normalización genera nombres de columnas repetidos."
        )

    return df.toDF(*nombres)


def tabla_existe(nombre):
    return spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{nombre}")


def cargados(nombre):
    if not tabla_existe(nombre):
        return set()

    return {
        fila["periodo_fuente"]
        for fila in (
            spark.table(f"{CATALOG}.{SCHEMA}.{nombre}")
            .select("periodo_fuente")
            .distinct()
            .collect()
        )
    }

def normalizar_periodo(valor):
    valor = valor.strip()

    if not re.fullmatch(r"\d{6}", valor):
        raise ValueError(f"Período inválido: {valor}")

    formato = "%Y%m" if valor.startswith("20") else "%m%Y"

    try:
        fecha = datetime.strptime(valor, formato)
    except ValueError as e:
        raise ValueError(f"Período inválido: {valor}") from e

    return fecha.strftime("%Y%m")

## 1. Inspeccion rapida 
Objetivo: confirmar columnas antes de cargar todo

In [ ]:
carpetas = sorted(d for d in glob.glob(f"{BASE}/viajes_stm/viajes_stm_*") if os.path.isdir(d))
print(len(carpetas), "carpetas de meses descomprimidas")
if carpetas:
    muestra = sorted(glob.glob(f"{carpetas[-1]}/**/*.csv", recursive=True))
    print("Archivos del ultimo mes:", [os.path.basename(m) for m in muestra])
    if muestra:
        df0 = leer_csv(muestra[:1])
        df0.printSchema()
        display(df0.limit(5))

## 2. Cargar viajes (mes por mes)

In [ ]:
TABLA_VIAJES = "viajes_stm"

carpetas = [
    p
    for p in glob.glob(f"{BASE}/viajes_stm/viajes_stm_*")
    if os.path.isdir(p)
]

lotes = {}

for carpeta in carpetas:
    codigo = os.path.basename(carpeta).replace("viajes_stm_", "")
    periodo = normalizar_periodo(codigo)

    if periodo in lotes:
        raise ValueError(
            f"Hay dos carpetas para el período {periodo}: "
            f"{lotes[periodo][1]} y {carpeta}. "
            "Revisarlas antes de cargar."
        )

    lotes[periodo] = (codigo, carpeta)

ya = cargados(TABLA_VIAJES)

for periodo in sorted(lotes):
    codigo, carpeta = lotes[periodo]

    if periodo in ya:
        print("Ya cargado:", periodo)
        continue

    csvs = sorted(
        p
        for p in glob.glob(f"{carpeta}/**/*", recursive=True)
        if os.path.isfile(p) and p.lower().endswith(".csv")
    )

    if not csvs:
        raise ValueError(f"No hay CSV en {carpeta}")

    print("Cargando:", periodo, "| Archivos:", len(csvs))

    df = (
        leer_csv(csvs)
        .withColumn("archivo_origen", F.lit(codigo))
        .withColumn("periodo_fuente", F.lit(periodo))
    )

    try:
        (
            df.write
            .format("delta")
            .mode("append")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLA_VIAJES}")
        )
    except Exception:
        print(f"FAILFAST fallo para {periodo}, reintentando con PERMISSIVE...")
        sep, enc = detectar_formato(csvs[0])
        df = (
            spark.read
            .option("header", True)
            .option("sep", sep)
            .option("encoding", enc)
            .option("inferSchema", False)
            .option("mode", "PERMISSIVE")
            .csv(csvs)
        )
        nombres = [limpiar_nombre(c) for c in df.columns]
        df = (
            df.toDF(*nombres)
            .withColumn("archivo_origen", F.lit(codigo))
            .withColumn("periodo_fuente", F.lit(periodo))
        )
        (
            df.write
            .format("delta")
            .mode("append")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLA_VIAJES}")
        )

    ya.add(periodo)

print("Carga de viajes finalizada.")

## 3. Cargar horarios

In [ ]:
FUENTES_HORARIOS = {
    "horarios_por_parada": {
        "uptu_pasada_variante.csv":
            "horarios_por_parada_uptu_pasada_variante",

        "uptu_pasada_circular.csv":
            "horarios_por_parada_uptu_pasada_circular",
    },

    "horarios_puntos_control": {
        "horarios_omnibus datos.csv":
            "horarios_puntos_control_horarios_omnibus_datos",
    },
}

for subcarpeta, esperados in FUENTES_HORARIOS.items():

    fechas = sorted(
        p
        for p in glob.glob(f"{BASE}/{subcarpeta}/*")
        if os.path.isdir(p)
        and re.fullmatch(r"\d{4}-\d{2}-\d{2}", os.path.basename(p))
    )

    if not fechas:
        raise ValueError(f"No hay descargas para {subcarpeta}")

    ultima = fechas[-1]

    encontrados = {}

    for path in glob.glob(f"{ultima}/**/*", recursive=True):
        if not os.path.isfile(path):
            continue

        if not path.lower().endswith(".csv"):
            continue

        archivo = os.path.basename(path).lower()

        if archivo in encontrados:
            raise ValueError(f"CSV repetido en la carpeta: {archivo}")

        encontrados[archivo] = path

    faltantes = set(esperados) - set(encontrados)
    inesperados = set(encontrados) - set(esperados)

    if faltantes or inesperados:
        raise ValueError(
            f"Revisar {ultima}. "
            f"Faltantes: {sorted(faltantes)}. "
            f"Inesperados: {sorted(inesperados)}."
        )

    # Preparar y verificar todos los CSV de esta fuente
    # antes de empezar a reemplazar sus tablas.
    preparados = []

    for archivo, tabla in esperados.items():
        path = encontrados[archivo]
        df = leer_csv([path])

        if not df.take(1):
            raise ValueError(f"CSV sin registros: {path}")

        df = df.withColumn(
            "archivo_origen",
            F.lit(os.path.basename(path))
        )

        preparados.append((tabla, df))

    for tabla, df in preparados:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"{CATALOG}.{SCHEMA}.{tabla}")
        )

        print("Tabla creada:", tabla, "| Descarga:", ultima)

# Inspeccion de tablas en Bronce

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

In [ ]:
tablas = [
    "viajes_stm",
    "horarios_por_parada_uptu_pasada_variante",
    "horarios_por_parada_uptu_pasada_circular",
    "horarios_puntos_control_horarios_omnibus_datos",
]

for tabla in tablas:
    nombre_completo = f"{CATALOG}.{SCHEMA}.{tabla}"

    print(f"\n──────── {nombre_completo} ────────")

    if spark.catalog.tableExists(nombre_completo):
        display(
            spark.table(nombre_completo).limit(10)
        )
    else:
        print("La tabla todavía no fue creada.")